In [3]:
using Cosmology
using Jens
using Jens.LensModel.ComLens: CombinedLens
using Jens.LensModel: SIS, Shear
using Jens.LightModel: PointImage
using Jens.LensGenerator: LensedPlane, LightPlane, GenGrid
using Jens.LensSystem: ForwardModel
using Jens.TimeDelay: LensTimeDelay, image_time_delays
using Jens.LensSolver: solve_images
using Jens.LensConstants: DAY_TO_SEC
using Jens.LensPointLikelihood: point_image_chi2, time_delay_chi2
using Jens.LensAdaptiveGrid: RefinementMap, adaptive_grid_info
using Statistics

## lens system

In [4]:
cosmo = Cosmology.FlatLCDM(0.7, 0.3, 0.0, 0.0)
z_lens, z_src = 0.3, 1.5
beta_x, beta_y = 0.05, -0.03

lens = CombinedLens(
    SIS   => (theta_E=1.0, xcentre=0.0, ycentre=0.0),
    Shear => (gamma1=0.05, gamma2=-0.02, xcentre=0.0, ycentre=0.0),
)
lp = LensedPlane(lens; z_lens=z_lens, cosmology=cosmo)

grid = GenGrid(pix_n=128, pix_size=0.09)
agn = PointImage(flux=100.0, beta_x=beta_x, beta_y=beta_y)
sys = ForwardModel(
    lens_plane   = lp,
    source_plane = LightPlane(agn; z=z_src),
    grid         = grid,
);

In [5]:
ref = RefinementMap(grid, lp; threshold=0.1, sub_n=4, z_source=z_src)
adaptive_grid_info(ref, grid)

AdaptiveGrid: 102/16641 pixels refined (0.6%)
  threshold: |detJ| < 0.1
  sub-sampling: 4×4
  total rays: 18171 (vs 16641 uniform)
  overhead: 9.2%


In [6]:
# Visual check: refined pixels should cluster around R ≈ θ_E = 1.0
idx = findall(ref.needs_refine)
r_values = [sqrt(grid.xg[i]^2 + grid.yg[i]^2) for i in idx]
println("Refined pixel radii:  min=$(round(minimum(r_values), digits=2))  " *
        "max=$(round(maximum(r_values), digits=2))  median=$(round(median(r_values), digits=2))")

Refined pixel radii:  min=0.72  max=0.92  median=0.81


### Effect of threshold

In [7]:
for thresh in [0.01, 0.05, 0.1, 0.2, 0.5]
    r = RefinementMap(grid, lp; threshold=thresh, sub_n=4, z_source=z_src)
    total = length(r.needs_refine)
    frac = r.n_refined / total * 100
    n_rays = (total - r.n_refined) + r.n_refined * 16
    overhead = round(n_rays / total * 100 - 100, digits=1)
    println("  threshold=$thresh  →  $(r.n_refined) pixels ($(round(frac,digits=2))%)  " *
            "overhead=$(overhead)%")
end

  threshold=0.01  →  10 pixels (0.06%)  overhead=0.9%
  threshold=0.05  →  48 pixels (0.29%)  overhead=4.3%
  threshold=0.1  →  102 pixels (0.61%)  overhead=9.2%
  threshold=0.2  →  218 pixels (1.31%)  overhead=19.7%
  threshold=0.5  →  890 pixels (5.35%)  overhead=80.2%


### Sub-pixel offsets structure

In [8]:
println("Offsets shape: $(size(ref.offsets))  — ($(ref.sub_n^2) sub-pixels, (dx, dy))")
println("Weights:       $(ref.weights)  — uniform 1/$(ref.sub_n^2)")
println()
println("First 5 offset pairs (in pixel units):")
for i in 1:min(5, size(ref.offsets, 1))
    println("  $(i): (dx=$(ref.offsets[i,1]), dy=$(ref.offsets[i,2]))")
end

Offsets shape: (16, 2)  — (16 sub-pixels, (dx, dy))
Weights:       [0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625, 0.0625]  — uniform 1/16

First 5 offset pairs (in pixel units):
  1: (dx=-0.375, dy=-0.375)
  2: (dx=-0.375, dy=-0.125)
  3: (dx=-0.375, dy=0.125)
  4: (dx=-0.375, dy=0.375)
  5: (dx=-0.125, dy=-0.375)
